# VideoStir MedVidQA Reproduction Notebook

This notebook reproduces the VideoStir pipeline on the MedVidQA dataset.

**Prerequisites:**
- Run this notebook from a GitHub checkout of VideoStir and install from its root with `!pip install -e .`.
- Attach Kaggle Datasets containing `train.json`, `val.json`, and `test.json`, plus the video files. Dataset files are not stored in the GitHub repository.
- Make the attached folders available at `./MedVidQA/` and `./videos/` (for example, with symlinks from `/kaggle/input/<dataset-folder>`).
- Download the Frame Reranker LoRA adapter from [Google Drive](https://drive.google.com/file/d/1CUC1i7zstZktWDp30Pts4s8E7QULDgns/view?usp=drive_link) and extract it to `./result/`.
- For the programmatic example, provide `./videos/lbPbM8018CE.webm` (video_id: lbPbM8018CE). Enable Kaggle internet access if retrieval needs to download missing videos or model weights.

In [ ]:
# Verify installation and check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

## 1. Quick Demo on a Single Sample

In [ ]:
# Run a quick demo using the CLI (this will show metrics on a random training sample)
!python -m videostir.eval demo --medvidqa-root MedVidQA --retrieval-dir medvidqa_retrieval

## 2. Run Retrieval on MedVidQA Samples

This runs the full VideoStir pipeline on the specified dataset. Videos will be downloaded from YouTube if not present locally.

**Note:** This requires the Frame Reranker adapter in `./result/` and takes several minutes per video.

In [ ]:
# Run retrieval on the validation set (first 5 samples)
!python -m videostir.eval retrieve \
    --dataset val \
    --num-samples 5 \
    --medvidqa-root MedVidQA \
    --video-root videos \
    --output-dir medvidqa_retrieval \
    --download-missing

## 3. Evaluate Retrieval Performance

In [ ]:
# Evaluate the retrieval results
!python -m videostir.eval evaluate \
    --dataset val \
    --num-samples 5 \
    --medvidqa-root MedVidQA \
    --output-dir eval_results \
    --retrieval-dir medvidqa_retrieval

## 4. Generate Detailed Reports

In [ ]:
# Generate text, markdown, and CSV reports
!python -m videostir.eval report \
    --results eval_results/eval_results.json \
    --output-dir eval_reports

## 5. Programmatic Usage (Python API)

You can also use the Python API directly for custom workflows.

In [ ]:
from videostir.inference import PipelineConfig, run_pipeline
from videostir.eval import MedVidQAEvaluator
import json

# Load a sample from MedVidQA that matches the available video
# The repo includes: videos/lbPbM8018CE.webm
with open("MedVidQA/train.json", "r") as f:
    train_data = json.load(f)

# Find sample with video_id lbPbM8018CE
sample = next(s for s in train_data if s["video_id"] == "lbPbM8018CE")
print(f"Sample ID: {sample['sample_id']}")
print(f"Video ID: {sample['video_id']}")
print(f"Question: {sample['question']}")
print(f"Answer time: {sample['answer_start_second']}s - {sample['answer_end_second']}s")

In [ ]:
# Run pipeline programmatically
video_path = f"videos/{sample['video_id']}.webm"
query = sample["question"]

config = PipelineConfig(
    video_path=video_path,
    query=query,
    output_dir="my_results",
    top_frames=128,
    frame_interval=30,
    top_k=3,
    spatial_k=3,
    batch_size=64,
)

result = run_pipeline(config)

print(f"Reranked frames: {len(result.reranked_frames)}")
print(f"Time focus frames: {len(result.time_focus_frames)}")
print(f"Intent: {result.intent}")

In [ ]:
# Evaluate the retrieved frames
evaluator = MedVidQAEvaluator(medvidqa_root="MedVidQA")

sample_metrics = evaluator.evaluate_sample(
    sample={
        "sample_id": sample["sample_id"],
        "video_id": sample["video_id"],
        "question": sample["question"],
        "answer": sample.get("answer", ""),
        "answer_start_second": sample["answer_start_second"],
        "answer_end_second": sample["answer_end_second"],
        "video_length": sample.get("video_length", 0),
    },
    retrieved_frames=result.reranked_frames
)

for k in ["mrr", "recall@10", "recall@25", "recall@50", "recall@128", "map@10", "coverage@128"]:
    print(f"{k:15s}: {sample_metrics[k]:.4f}")

## 6. Batch Processing Multiple Videos

Use `run_batch_from_config` to process multiple videos with a JSON config.

In [ ]:
# Create a batch config JSON
batch_config = [
    {
        "id": "sample_1",
        "video_path": "videos/lbPbM8018CE.webm",
        "query": "How to perform epley maneuver for vertigo?"
    }
]

import json
with open("batch_config.json", "w") as f:
    json.dump(batch_config, f, indent=2)

# Run batch (uncomment to execute)
# from videostir.inference import run_batch_from_config
# results = run_batch_from_config(
#     config_path="batch_config.json",
#     output_root="batch_output",
#     video_root="."
# )

## 7. CLI Reference

```bash
# Single video retrieval
python -m videostir.inference run --video videos/lbPbM8018CE.webm --query "Show me the medical procedure" --output results/

# Batch processing
python -m videostir.inference batch --input batch_config.json --output results/

# Interactive Q&A with Qwen2.5-VL-3B
python -m videostir.inference answer --video videos/lbPbM8018CE.webm --output results/

# MedVidQA Evaluation CLI
python -m videostir.eval info          # Dataset statistics
python -m videostir.eval retrieve    # Run retrieval on dataset
python -m videostir.eval evaluate    # Evaluate cached results
python -m videostir.eval report      # Generate reports
python -m videostir.eval demo        # Quick demo on random sample
```